# 23 — Suitability + Skill Gaps

This notebook prototypes the first end-to-end Chapter 3 positioning flow using frozen upstream artefacts and a validated UserProfile schema.

What was implemented and validated here:
- Candidate set construction via hard filters derived from `build_user_profile()`.
- Skill-match suitability component using cosine similarity in the shared PCA skill space, with a normalized variant used only for aggregation.
- Salary alignment component defined as a one-sided score: meeting/exceeding the target salary does not reduce suitability.
- A simple weighted suitability score combining normalized skill fit and salary alignment (baseline weights: `w_skill=0.7`, `w_salary=0.3`).
- Skill gap analysis computed over the top-K most suitable jobs using the Chapter 1 skill probability matrix (`{skill}_prob` columns), producing a user-level diagnostic ranked by gap severity.

Key design decisions:
- Normalization is applied only to combine suitability components; raw skill vectors remain the source of truth for gap analysis.
- Candidate selection is strictly filter-based and contains no scoring logic.
- Gap analysis is aggregated over top-K suitable jobs and intentionally does not depend on job IDs in the final gap report.


## Set up

### Libraries

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#===
import sys
from pathlib import Path

### Path

In [2]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

In [3]:
from src.job_intel.schemas import build_user_profile
from src.job_intel.config import CH2_PROCESSED_DF, SKILL_PROB_MATRIX

### Data

In [4]:
df = pd.read_csv(CH2_PROCESSED_DF)
df.head()

,job_id,Job Description,Rating,Size,Founded,Industry,Sector,role_source,state,ownership_clean,...,skill_PC1,skill_PC2,skill_PC3,skill_PC4,skill_PC5,skill_PC6,skill_PC7,skill_PC8,skill_PC9,skill_PC10
0,0,"ABOUT HOPPER\n\nAt Hopper, we’re on a mission ...",3.5,501 to 1000 employees,2007.0,Travel Agencies,Travel & Tourism,data_scientist,NY,private,...,0.462016,0.107793,-0.286497,-0.961280,0.050588,-0.055616,0.395975,0.480314,0.016349,-0.239943
1,1,"At Noom, we use scientifically proven methods ...",4.5,1001 to 5000 employees,2008.0,"Health, Beauty, & Fitness",Consumer Services,data_scientist,NY,private,...,0.018662,-0.339362,0.631396,-0.972227,-0.139904,-0.183168,0.286340,0.235295,0.125379,0.213347
2,2,Decode_M\n\nhttps://www.decode-m.com/\n\nData ...,NaN,1 to 50 employees,NaN,Unknown,Unknown,data_scientist,NY,unknown,...,-0.485473,-0.143990,0.604327,0.428753,0.363498,-0.395078,0.811758,0.003043,0.025900,-0.188561
3,3,Sapphire Digital seeks a dynamic and driven mi...,3.4,201 to 500 employees,2019.0,Internet,Information Technology,data_scientist,NJ,private,...,0.009683,1.123316,0.112544,-0.535952,-0.301241,-0.235044,0.295499,-0.211524,0.196350,-0.283105
4,4,"Director, Data Science - (200537)\nDescription...",3.4,51 to 200 employees,2007.0,Advertising & Marketing,Business Services,data_scientist,NY,private,...,-0.502115,-0.138427,0.758690,-0.065791,-0.458638,-0.280799,-0.722739,-0.690126,0.256452,-0.619181


## Build user profile instance

In [5]:
skill_text = df.loc[df['job_id'] == 123, 'Job Description'].iloc[0]
skill_text

'Grubhub is dedicated to connecting hungry diners with our wide network of restaurants across the country. Our innovative technology, easy-to-use platforms and streamlined delivery capabilities make us an industry leader today, and in the future of online food ordering.\nWe strive to create a workplace that reflects the diversity of our customers and the communities we serve. When you join our team, you become part of a community that works together to innovate, solve problems, take risks, grow, work hard and have a ton of fun in the process!\nWhy Work For Us\nWe have a fast-paced environment and that is what our teams thrive on. Grubhub believes in empowering people and offering opportunities for development, as well as professional growth. We value strong, positive relationships in all areas: with each other, our customers and our greater community. Want to be a part of a team of diverse collaborators in an authentically fun culture? If so, we want to talk to you - and hear what’s yo

In [6]:
profile = build_user_profile( skill_text=skill_text,
                                    current_state='CA',
                                    job_title_family  = "data_scientist",
                                    job_title_rich  = 'ML_AI_data_scientist',
                                    target_sectors = "Information Technology",
                                    salary_target= 150000,
                                    explain_skills=True)

In [7]:
profile["raw_inputs"]


{'skill_text': 'Grubhub is dedicated to connecting hungry diners with our wide network of restaurants across the country. Our innovative technology, easy-to-use platforms and streamlined delivery capabilities make us an industry leader today, and in the future of online food ordering.\nWe strive to create a workplace that reflects the diversity of our customers and the communities we serve. When you join our team, you become part of a community that works together to innovate, solve problems, take risks, grow, work hard and have a ton of fun in the process!\nWhy Work For Us\nWe have a fast-paced environment and that is what our teams thrive on. Grubhub believes in empowering people and offering opportunities for development, as well as professional growth. We value strong, positive relationships in all areas: with each other, our customers and our greater community. Want to be a part of a team of diverse collaborators in an authentically fun culture? If so, we want to talk to you - and

In [8]:
profile["derived"]["skill_pcs"].shape

(1, 10)

### Apply filter

In [9]:
if profile['raw_inputs']['current_state'] is not None:
    filter_step1 = df[df['state'] == profile['raw_inputs']['current_state']]
else:
    filter_step1 = df

if profile['raw_inputs']['job_title_family'] is not None:
    filter_step2 = filter_step1[filter_step1['job_title_family'] == profile['raw_inputs']['job_title_family']]
else:
    filter_step2 = filter_step1

if profile['raw_inputs']['job_title_rich'] is not None:
    filter_step3 = filter_step2[filter_step2['title_rich'] == profile['raw_inputs']['job_title_rich']]
else:
    filter_step3 = filter_step2

if profile['raw_inputs']['target_sectors'] is not None:
    out = filter_step3[filter_step3['Sector'].isin(profile['raw_inputs']['target_sectors'])]
else:
    out = filter_step3

print(f'Initial number of jobs available: {len(df)}.')
print(f'Current number of jobs available: {len(out)}')

if len(out) == 0:
    raise ValueError('No jobs available within the current constraints. Please widen your filters')


Initial number of jobs available: 6161.
Current number of jobs available: 10


### Wrap that into a function

In [10]:
from typing import Optional, Union, List, Any

def candidate_set_construction(
    df: pd.DataFrame,
    skill_text: str = "",
    current_state: Optional[str] = "ALL",
    job_title_family: Optional[str] = None,
    job_title_rich: Optional[str] = None,
    target_sectors: Optional[Union[str, List[str]]] = None,
    salary_target: Optional[Union[int, float, str]] = None,
    explain_skills: bool = False,
):
    profile = build_user_profile(
        skill_text=skill_text,
        current_state=current_state,
        job_title_family=job_title_family,
        job_title_rich=job_title_rich,
        target_sectors=target_sectors,
        salary_target=salary_target,
        explain_skills=explain_skills,
    )

    out = df

    # 1) state
    if profile["raw_inputs"]["current_state"] is not None:
        out = out[out["state"] == profile["raw_inputs"]["current_state"]]

    # 2) sectors
    if profile["raw_inputs"]["target_sectors"] is not None:
        out = out[out["Sector"].isin(profile["raw_inputs"]["target_sectors"])]

    # 3) title_rich
    if profile["raw_inputs"]["job_title_rich"] is not None:
        out = out[out["title_rich"] == profile["raw_inputs"]["job_title_rich"]]

    # 4) title_family
    if profile["raw_inputs"]["job_title_family"] is not None:
        out = out[out["job_title_family"] == profile["raw_inputs"]["job_title_family"]]

    print(f"Initial number of jobs available: {len(df)}.")
    print(f"Current number of jobs available: {len(out)}.")

    if out.empty:
        raise ValueError("No jobs available within the current constraints. Please widen your filters.")

    return profile, out


## Suitability - skill-match score

In [11]:
profile, candidates_df = candidate_set_construction( df = df,
                                                    skill_text=skill_text,
                                                    current_state='CA',
                                                    job_title_family  = "data_scientist",
                                                    job_title_rich  = 'ML_AI_data_scientist',
                                                    target_sectors = "Information Technology",
                                                    salary_target= 150000,
                                                    explain_skills=True)

Initial number of jobs available: 6161.
Current number of jobs available: 10.


### Cosine similarity across the PC space

In [12]:
pc_cols = profile["derived"]["skill_pcs"].columns.tolist()

user_pcs_df = profile["derived"]["skill_pcs"] 
job_pcs_df = candidates_df[pc_cols]   

user_vec = user_pcs_df.to_numpy() 
job_mat = job_pcs_df.to_numpy() 


In [13]:
from sklearn.metrics.pairwise import cosine_similarity
candidates_df['skill_match_score'] = cosine_similarity(user_vec, job_mat).flatten() 

s = candidates_df["skill_match_score"]
candidates_df["skill_match_norm"] = ((s + 1) / 2).clip(0, 1)


### Return top 10 jobs matching current skills

In [14]:
candidates_df[['title_rich', 'state', 'Sector', 'sal_mean', 'skill_match_norm']].sort_values(by = 'skill_match_norm', ascending=False).head(10)

,title_rich,state,Sector,sal_mean,skill_match_norm
3006,ML_AI_data_scientist,CA,Information Technology,225000.0,0.795370
3055,ML_AI_data_scientist,CA,Information Technology,153500.0,0.795370
2992,ML_AI_data_scientist,CA,Information Technology,225000.0,0.678548
3046,ML_AI_data_scientist,CA,Information Technology,153500.0,0.661658
3192,ML_AI_data_scientist,CA,Information Technology,140500.0,0.614623
2994,ML_AI_data_scientist,CA,Information Technology,225000.0,0.540212
3149,ML_AI_data_scientist,CA,Information Technology,156500.0,0.502365
3029,ML_AI_data_scientist,CA,Information Technology,174500.0,0.485064
2887,ML_AI_data_scientist,CA,Information Technology,161000.0,0.461954
3146,ML_AI_data_scientist,CA,Information Technology,156500.0,0.287556


## Suitability - salary alignment

In [15]:
target = profile["raw_inputs"]["salary_target"]
if target is None:
    candidates_df["salary_score"] = 1.0
else:
    candidates_df["salary_score"] = (candidates_df["sal_mean"] / target).clip(upper=1)


w_skill = 0.7
w_salary = 0.3

candidates_df['suitability'] = w_skill * candidates_df['skill_match_norm'] + w_salary * candidates_df['salary_score']


In [16]:
candidates_df[['title_rich', 'state', 'Sector', 'sal_mean', 'skill_match_score', 'skill_match_norm', 'salary_score', 'suitability']].sort_values(by = 'suitability', ascending=False).head(10)

,title_rich,state,Sector,sal_mean,skill_match_score,skill_match_norm,salary_score,suitability
3006,ML_AI_data_scientist,CA,Information Technology,225000.0,0.590740,0.795370,1.000000,0.856759
3055,ML_AI_data_scientist,CA,Information Technology,153500.0,0.590740,0.795370,1.000000,0.856759
2992,ML_AI_data_scientist,CA,Information Technology,225000.0,0.357095,0.678548,1.000000,0.774983
3046,ML_AI_data_scientist,CA,Information Technology,153500.0,0.323316,0.661658,1.000000,0.763161
3192,ML_AI_data_scientist,CA,Information Technology,140500.0,0.229245,0.614623,0.936667,0.711236
2994,ML_AI_data_scientist,CA,Information Technology,225000.0,0.080424,0.540212,1.000000,0.678148
3149,ML_AI_data_scientist,CA,Information Technology,156500.0,0.004730,0.502365,1.000000,0.651656
3029,ML_AI_data_scientist,CA,Information Technology,174500.0,-0.029871,0.485064,1.000000,0.639545
2887,ML_AI_data_scientist,CA,Information Technology,161000.0,-0.076092,0.461954,1.000000,0.623368
3146,ML_AI_data_scientist,CA,Information Technology,156500.0,-0.424888,0.287556,1.000000,0.501289


In [17]:
candidates_df[["skill_match_norm","salary_score","suitability"]].describe()


,skill_match_norm,salary_score,suitability
count,10.000000,10.000000,10.000000
mean,0.582272,0.993667,0.705690
std,0.158597,0.020028,0.110749
min,0.287556,0.936667,0.501289
25%,0.489390,1.000000,0.642573
50%,0.577417,1.000000,0.694692
75%,0.674325,1.000000,0.772028
max,0.795370,1.000000,0.856759


## Skill gap analysis

In [ ]:
skill_prob_matrix = pd.read_csv(SKILL_PROB_MATRIX)


In [25]:
# skill_prob_matrix already loaded above
# skill_prob_matrix = pd.read_csv(SKILL_PROB_MATRIX)

user_skills = profile["derived"]["skill_vector"]
skill_cols = user_skills.columns.tolist()

target_k = 10
k = min(len(candidates_df), target_k)

top_k_jobs = candidates_df.sort_values("suitability", ascending=False).head(k)

# --- use prob columns instead of binary flags ---
prob_cols = [f"{s}_prob" for s in skill_cols]

top_k_probs = (
    skill_prob_matrix
    .loc[skill_prob_matrix["job_id"].isin(top_k_jobs["job_id"]), ["job_id"] + prob_cols]
)

job_skill_rate = (
    top_k_probs[prob_cols]
    .mean(axis=0)
    .rename("job_skill_rate")
    .reset_index()
)

# convert "skillname_prob" -> "skillname"
job_skill_rate["skill"] = job_skill_rate["index"].str.replace("_prob", "", regex=False)
job_skill_rate = job_skill_rate.drop(columns="index")

job_skill_rate["user_skill"] = user_skills.T.iloc[:, 0].values

job_skill_rate["skill_gap"] = (
    (job_skill_rate["user_skill"] == 0)
    * job_skill_rate["job_skill_rate"]
)

# optional: view top gaps
job_skill_rate.sort_values("skill_gap", ascending=False)


,job_skill_rate,skill,user_skill,skill_gap
8,6.005704e-01,ml_ai__advanced,0,6.005704e-01
7,5.784649e-01,ml_ai__intermediate,0,5.784649e-01
6,4.283806e-01,ml_ai__basic,0,4.283806e-01
13,3.301483e-01,bi_viz__intermediate,0,3.301483e-01
5,3.208742e-01,data_engineering_pipelines__advanced,0,3.208742e-01
25,3.192880e-01,soft_skills__leadership,0,3.192880e-01
26,2.497175e-01,domain_specific__none,0,2.497175e-01
12,2.000786e-01,bi_viz__basic,0,2.000786e-01
16,2.752245e-02,cloud__intermediate,0,2.752245e-02
11,1.239308e-02,analytics_stats__advanced,0,1.239308e-02


# == End of Notebook == 